# Analitycal model of Google's MICROSERVICES DEMO

This notebook requires some json file to build the jackson network and solve the traffic equations. In particular we will need:

1. overall_avg_metrics-low_load.json collected in low load to estimate service rates for leaf services.
2. routing_count.json for building the routing matrix and solving the traffic equations.
3. overall_avg_metrics-normal_load.json to measure empirical metrics under experimental load for comparisons with model predictions.
4. upstream_response_time-low_load.json to estimate service rates for upstream nodes.
5. upstream_response_time-normal_load.json to measure the response time of upstream services under experimental arrival rates.

The notebook will be able to conclude whether with the currenct external arrival rate the system will be able to reach a steady state assuming M/M/1 queueing behavior for all nodes. In the case a steady state is reachable the notebook will compute staedy state performance metrics and compare them with actual measurements.


## Estimating service rates
Since we have not collected CPU consumption metrics (that allow us to calculate the service rate $\mu$ as throughput X divided by CPU utilization U), we will need to estimate it by collecting the average response time in low load scenarios, where the service time can be approximated as the response time (waiting time negligible). However, in the metrics we collected, the response time of upstream services account for the response time of downstream services. So, for upstream metrics we extract their average service time from collected traces instead of the collected metrics.

$R_i \approx S_i$, then $\mu_i \approx \frac{1}{R_i}$


In [35]:
# load json data
import json
import numpy as np

with open("../jupyter_data/overall_avg_metrics-low_load.json", "r") as f:
    metrics_dict = json.load(f)

with open("../jupyter_data/routing_count.json","r") as f:
    routing_dict=json.load(f)

with open("../jupyter_data/upstream_response_time-low_load.json", "r") as f:
    upstream_resp_dict=json.load(f)

other_endpoints = [
    "GetAds",
    "ListRecommendations",
    "GetCart",
    "AddItem",
    "EmptyCart",
    "PlaceOrder",
    "Charge",
    "GetQuote",
    "ShipOrder",
    "SendOrderConfirmation",
    "GetSupportedCurrencies",
    "Convert",
    "ListProducts",
    "GetProduct",
]
services = [
    "frontend",
    "adservice",
    "recommendationservice",
    "cartservice",
    "checkoutservice",
    "paymentservice",
    "shippingservice",
    "emailservice",
    "currencyservice",
    "productcatalogservice",
]

metrics = [
    "requests_total",
    "requests_duration",
    "active_requests",
]

In [36]:
# Compute approximated service rate and save it in a json file

response_time = {service: 0 for service in services}

for service, metric_dict in metrics_dict.items():
    
    if service in ["frontend", "checkoutservice", "recommendationservice"]:
        R_i=upstream_resp_dict[service]
    else:
        R_i = metric_dict["requests_duration"]

    mu_i = 1 / R_i
    response_time[service] = mu_i

with open("../jupyter_data/service_rates.json", "w") as f:
    json.dump(response_time, f, indent=4)

In [37]:
# Run to show estimated service rate for each microservice
print("Service rates for each service:")

for service, rate in response_time.items():
    print(f"{service} - {rate}")

Service rates for each service:
frontend - 1081.9230000716905
adservice - 2864.3147896879223
recommendationservice - 3429.7237736846037
cartservice - 631.6855044437549
checkoutservice - 3543.131423910748
paymentservice - 3052.6315789473674
shippingservice - 11936.212286589074
emailservice - 3334.1506242890227
currencyservice - 4596.678529062873
productcatalogservice - 15338.68430627322


## Solving Traffic Equations

Given the routing count of departing requests on each node of the Queueing network (used to compute routing probabilities) and the external arrival rate $\gamma$ for the frontend, we can solve the Traffic Equations and obtain the aggregate arrival rate $\lambda_i$ for each service i. We then conclude if the system is stable for the current arrival rate.

$\lambda_i = \gamma_i + \sum_{j=0}^{N-1}q_{ji}\lambda_j$ 

In the matrix form we have:

If $\underline{\lambda}=[\lambda_1, \lambda_2, ..., \lambda_N]^T$ and $\underline{\gamma}=[\gamma_1, \gamma_2, ..., \gamma_N]^T$, then $\underline{\lambda}=\underline{\gamma}+Q^T\underline{\lambda}$ which implies $(I-Q^T)\underline{\lambda}=\underline{\gamma}$

In [38]:
# Run to build the external arrival rate vector
service_index_map = {
        "frontend": 0,
        "adservice": 1,
        "recommendationservice": 2,
        "cartservice": 4,
        "checkoutservice": 8,
        "paymentservice": 7,
        "shippingservice": 5,
        "emailservice": 9,
        "currencyservice": 6,
        "productcatalogservice": 3,
    }

function_map = {
    "GetAds": "adservice",
    "ListRecommendations": "recommendationservice",
    "GetCart": "cartservice",
    "AddItem": "cartservice",
    "EmptyCart": "cartservice",
    "PlaceOrder": "checkoutservice",
    "Charge": "paymentservice",
    "GetQuote": "shippingservice",
    "ShipOrder": "shippingservice",
    "SendOrderConfirmation": "emailservice",
    "GetSupportedCurrencies": "currencyservice",
    "Convert": "currencyservice",
    "ListProducts": "productcatalogservice",
    "GetProduct": "productcatalogservice",
}
# Set the value you see from the collected metrics on the frontend in the empirical experiment
EXTERNAL_ARRIVAL_RATE=11.4

gamma=[0.0 for _ in range(10)]
gamma[0]=EXTERNAL_ARRIVAL_RATE
gamma=np.array(gamma).reshape(-1,1)


In [39]:
# Run to show gamma
print("External arrival rates for each service:")

for key, value in service_index_map.items():
        print(f"{key} - {gamma[value][0]}")

External arrival rates for each service:
frontend - 11.4
adservice - 0.0
recommendationservice - 0.0
cartservice - 0.0
checkoutservice - 0.0
paymentservice - 0.0
shippingservice - 0.0
emailservice - 0.0
currencyservice - 0.0
productcatalogservice - 0.0


In [40]:
# Run to build the routing matrix
Q=np.array([[0.0 for _ in range(10)] for _ in range(10)])

for service in services:
    total=0
    for name, value in routing_dict[service].items():
        if name=="in":
            continue
        if name=="out":
            total+=value
            continue
        total+=value
        Q[service_index_map[service],service_index_map[function_map[name]]]+=value
    Q[service_index_map[service],:]=Q[service_index_map[service],:]/total

In [41]:
# Run to show Q
print("Full routing matrix:")

for key, value in service_index_map.items():
        print(f"{key} - {value}")
print()
print("\t", end="")
for i in range(10):
    print(f"{i}", end="\t")
print()
for i in range(10):
    print(i, end="\t")
    for j in range(10):
        print(f"{Q[i,j]:.3f}", end="\t")
    print()

Full routing matrix:
frontend - 0
adservice - 1
recommendationservice - 2
cartservice - 4
checkoutservice - 8
paymentservice - 7
shippingservice - 5
emailservice - 9
currencyservice - 6
productcatalogservice - 3

	0	1	2	3	4	5	6	7	8	9	
0	0.000	0.000	0.000	0.502	0.032	0.000	0.342	0.000	0.031	0.000	
1	0.000	0.000	0.000	0.000	0.000	0.000	0.000	0.000	0.000	0.000	
2	0.000	0.000	0.000	1.000	0.000	0.000	0.000	0.000	0.000	0.000	
3	0.000	0.088	0.000	0.671	0.054	0.040	0.147	0.000	0.000	0.000	
4	0.000	0.000	0.198	0.033	0.000	0.000	0.602	0.000	0.000	0.033	
5	0.000	0.000	0.000	0.000	0.125	0.000	0.875	0.000	0.000	0.000	
6	0.000	0.051	0.133	0.122	0.194	0.010	0.409	0.010	0.000	0.000	
7	0.000	0.000	0.000	0.000	0.000	1.000	0.000	0.000	0.000	0.000	
8	0.000	0.000	0.000	0.000	1.000	0.000	0.000	0.000	0.000	0.000	
9	0.000	0.000	1.000	0.000	0.000	0.000	0.000	0.000	0.000	0.000	


In [42]:
# Run to solve for lambda and print results

aggregate_arrival_rates = np.linalg.solve(np.eye(10) - Q.T, gamma)

print("Aggregate arrival rates for each service:")

for key, value in service_index_map.items():
        print(f"{key} - {aggregate_arrival_rates[value][0]}")

Aggregate arrival rates for each service:
frontend - 11.4
adservice - 6.440774050551979
recommendationservice - 7.129069519628317
cartservice - 10.693482801244617
checkoutservice - 0.3518008610033673
paymentservice - 0.3518008610033671
shippingservice - 2.823153318272877
emailservice - 0.3518008610033672
currencyservice - 34.90530241677677
productcatalogservice - 53.07674012190443


In [43]:
# Run to check if the system can actually handle the load assuming M/M/1 queueing behavior for all nodes.
mu=np.zeros((10,1),float)

for service, rsp_time in response_time.items():
    mu[service_index_map[service],0]=rsp_time

for i in range(10):
    if aggregate_arrival_rates[i,0]>mu[i,0]:
        print(f"Steady state does not exist, in node {i} lambda is bigger than mu: {aggregate_arrival_rates[i,0]}>{mu[i,0]}")

print("Steady state is reachable, we can now proceed to compute the steady state performance of the system.")

Steady state is reachable, we can now proceed to compute the steady state performance of the system.


## Compute Performance Metrics

We now compute performance metrics at steady state considering all nodes behaving as M/M/1 queues:

- Mean queue length at node i: $L_i=\frac{\rho_i}{1-\rho_i}$, with $\rho_i=\frac{\lambda_i}{\mu_i}$
- Mean system queue length: $L=L_1+L_2+...+L_N=\sum_{i=0}^{N-1}\frac{\rho_i}{1-\rho_i}$
- Mean system response time: $W=\frac{L}{\gamma}$, where $\gamma=\sum_{i=0}^{N-1}\gamma_i$
- Mean response time at node i: $W_i=\frac{L_i}{\lambda_i}$

In [44]:
# Run to compute utilization for every node and then print it
rho=np.divide(aggregate_arrival_rates, mu)
print("Utilization rate for each service::")
for key, value in service_index_map.items():
        print(f"{key} - {rho[value][0]}")

Utilization rate for each service::
frontend - 0.010536794207392404
adservice - 0.0022486264686199958
recommendationservice - 0.0020786133199203532
cartservice - 0.016928491671913553
checkoutservice - 9.929094321177211e-05
paymentservice - 0.00011524510963903408
shippingservice - 0.00023652003252697085
emailservice - 0.00010551438751462692
currencyservice - 0.007593592241894916
productcatalogservice - 0.003460318959703544


In [45]:
# Run to compute mean queue length for every node and the mean system queue length, then print them
L=np.divide(rho,1-rho)
L_tot=np.sum(L)
print("Mean queue length for each service::")
for key, value in service_index_map.items():
        print(f"{key} - {L[value][0]}")
print(f"Mean number of requests in the system: ")
print(L_tot)

Mean queue length for each service::
frontend - 0.010649000534539255
adservice - 0.002253694185016599
recommendationservice - 0.002082942952886858
cartservice - 0.01722000030364414
checkoutservice - 9.930080288215197e-05
paymentservice - 0.00011525839260512667
shippingservice - 0.00023657598748722642
emailservice - 0.00010552552197544532
currencyservice - 0.007651696102052802
productcatalogservice - 0.003472334344068755
Mean number of requests in the system: 
0.04388632912715836


In [46]:
# Run to compute mean system response time and print the results
gamma_tot=np.sum(gamma)
W_tot=L_tot/gamma_tot
print("Mean system response time: ")
print(W_tot)

Mean system response time: 
0.0038496779936103825


In [47]:
# Run to compute response time at node i and show the results
W=np.divide(L,aggregate_arrival_rates)
print("Mean response time for each service:: ")
for key, value in service_index_map.items():
        print(f"{key} - {W[value][0]}")

Mean response time for each service:: 
frontend - 0.0009341228539069522
adservice - 0.000349910455999222
recommendationservice - 0.0002921759911517113
cartservice - 0.0016103266469592022
checkoutservice - 0.00028226424062447505
paymentservice - 0.0003276239639561983
shippingservice - 8.379849084213277e-05
emailservice - 0.00029995811174104916
currencyservice - 0.00021921300124233035
productcatalogservice - 6.542101749454928e-05


## Comparisons with empirically collected data

Like previously said, the metrics we have collected for upstream services also comprehend downstream contributions. For example in the frontend service the mean duration is just the response time for the entire system, while the mean active requests represents the mean user number in the system. We will derive the corresponding empirical metrics in the following way:

- Empirical Average System Response Time: just the mean response time at the frontend.
- Empirical Average Number of Requests In The System: just the mean active requests number in the frontend.
- Empirical Average System Arrival Rate: just the mean number of arrival of requests per second in the frontend.
- Empirical Response Time At Service i: for leaf services they correspond to their measured local response time, while for upstream services we compute them from traces.
- Empirical Average Service Arrival Rates: we already got that as the local arrival rate at each service.
- Empirical Queue Length at Service i: estimated using Little's Law on the singular queues for upstream services, while for leaf services it corresponds to their mean active requests metric.


In [52]:
# Run to see the empirical average system arrival rate at steady state
with open("../jupyter_data/overall_avg_metrics-normal_load.json", "r") as f:
    normal_metrics_dict=json.load(f)

with open("../jupyter_data/upstream_response_time-normal_load.json", "r") as f:
    normal_upstream_resp_dict=json.load(f)

print(f"Average system arrival rate: {normal_metrics_dict["frontend"]["requests_rate"]}")
print(f"System arrival rate used in the model: {np.sum(gamma)}")

Average system arrival rate: 11.409104094918646
System arrival rate used in the model: 11.4


In [53]:
# Run to see the empirical average number of requests in the system

print(f"Average system queue length {normal_metrics_dict["frontend"]["active_requests"]}")
print(f"Predicted system queue length: {L_tot}")

Average system queue length 0.017807616595117887
Predicted system queue length: 0.04388632912715836


In [54]:
# Run to see the empirical average system response time

print(f"Average system response time {normal_metrics_dict["frontend"]["requests_duration"]}")
print(f"Predicted system response time: {W_tot}")

Average system response time 0.037833627839193654
Predicted system response time: 0.0038496779936103825


In [ ]:
# Run to compute and show the empirical response time for each node i

response_time = {service: 0 for service in services}

for service, metric_dict in normal_metrics_dict.items():
    
    if service in ["frontend", "checkoutservice", "recommendationservice"]:
        R_i=normal_upstream_resp_dict[service]
    else:
        R_i = metric_dict["requests_duration"]

    response_time[service] = R_i

print("Response time at each service node compared with model prediction")

W_emp=np.zeros(10,float)

for service, rsp_time in response_time.items():
    print(f"{service}\t\t predicted response time: {W[service_index_map[service]][0]}, actual response time: {rsp_time}")
    W_emp[service_index_map[service]]=rsp_time

Response time at each service node compared with model prediction
frontend	 predicted response time: 0.0009341228539069522, actual response time: 0.00048268710165844103
adservice	 predicted response time: 0.000349910455999222, actual response time: 0.00017635606511275277
recommendationservice	 predicted response time: 0.0002921759911517113, actual response time: 0.0004331904648836431
cartservice	 predicted response time: 0.0016103266469592022, actual response time: 0.0009173713474526827
checkoutservice	 predicted response time: 0.00028226424062447505, actual response time: 0.0005086384072209036
paymentservice	 predicted response time: 0.0003276239639561983, actual response time: 0.0002908912830558278
shippingservice	 predicted response time: 8.379849084213277e-05, actual response time: 0.00012674581612274753
emailservice	 predicted response time: 0.00029995811174104916, actual response time: 0.0003632991771979872
currencyservice	 predicted response time: 0.00021921300124233035, actual 

In [ ]:
# Run to show the empirical arrival rate for each node
lambda_emp=np.zeros(10,float)
print("Arrival rate at each service node compared with model prediction")
for service in services:
    print(f"{service}\t\t predicted arrival rate: {aggregate_arrival_rates[service_index_map[service]][0]}, actual arrival rate: {normal_metrics_dict[service]["requests_rate"]}")
    lambda_emp[service_index_map[service]]=normal_metrics_dict[service]["requests_rate"]

Arrival rate at each service node compared with model prediction
frontend	 predicted arrival rate: 11.4, actual arrival rate: 11.409104094918646
adservice	 predicted arrival rate: 6.440774050551979, actual arrival rate: 6.420271368821532
recommendationservice	 predicted arrival rate: 7.129069519628317, actual arrival rate: 7.135381278827655
cartservice	 predicted arrival rate: 10.693482801244617, actual arrival rate: 10.663037216679964
checkoutservice	 predicted arrival rate: 0.3518008610033673, actual arrival rate: 0.3546247207915764
paymentservice	 predicted arrival rate: 0.3518008610033671, actual arrival rate: 0.3534608083915769
shippingservice	 predicted arrival rate: 2.823153318272877, actual arrival rate: 2.8497575913378994
emailservice	 predicted arrival rate: 0.3518008610033672, actual arrival rate: 0.3530665491005801
currencyservice	 predicted arrival rate: 34.90530241677677, actual arrival rate: 34.9148335884104
productcatalogservice	 predicted arrival rate: 53.0767401219044

In [ ]:
# Run to compute and show empirical queue length for each node

L_emp=np.zeros(10,float)

for service, metric_dict in normal_metrics_dict.items():
    
    if service in ["frontend", "checkoutservice", "recommendationservice"]:
        L_i = normal_metrics_dict[service]["requests_rate"]*response_time[service]
    else:
        L_i = metric_dict["active_requests"]

    response_time[service] = L_i
    L_emp[service_index_map[service]]=L_i

print("Response time at each service node compared with model prediction")

for service in services:
    print(f"{service}\t\t predicted queue length: {L[service_index_map[service]][0]}, actual queue length: {response_time[service]}")

Response time at each service node compared with model prediction
frontend	 predicted queue length: 0.010649000534539255, actual queue length: 0.005507027388095733
adservice	 predicted queue length: 0.002253694185016599, actual queue length: 4.203907830274662e-45
recommendationservice	 predicted queue length: 0.002082942952886858, actual queue length: 0.0030909791332973955
cartservice	 predicted queue length: 0.01722000030364414, actual queue length: 2.557906589118888e-57
checkoutservice	 predicted queue length: 9.930080288215197e-05, actual queue length: 0.00018037575314458512
paymentservice	 predicted queue length: 0.00011525839260512667, actual queue length: 0.0
shippingservice	 predicted queue length: 0.00023657598748722642, actual queue length: 0.0
emailservice	 predicted queue length: 0.00010552552197544532, actual queue length: 0.0
currencyservice	 predicted queue length: 0.007651696102052802, actual queue length: 0.0
productcatalogservice	 predicted queue length: 0.003472334344